In [45]:
import src.utils.database_utils as db_utils
import src.utils.dictionaries as mydict
import src.utils.connection_utils as con_utils

import pandas as pd
import time
from tqdm import tqdm
import requests
import json
import ast
import numpy as np
from sqlalchemy import text

In [ ]:
from sqlalchemy import text

def update_alcohol_label(flights, spots):
    for index, flight in flights.iterrows():
        # Filter spots for the current flight's advertiser, product, and date range
        flight_spots = spots[
            (spots['advertiser'] == flight['advertiser']) &
            (spots['product'] == flight['product']) &
            (spots['date'] >= flight['start_date']) &
            (spots['date'] <= flight['end_date'])
        ]

        # Check if all rows for alcohol_kansspelen are True (1)
        if len(flight_spots) > 0 and flight_spots['alcohol_kansspelen'].all():
            flights.at[index, 'alcohol_kansspelen'] = 1  # Mark as True/1
        else:
            flights.at[index, 'alcohol_kansspelen'] = 0  # Mark as False/0

def update_alcohol_column(flights_df, engine):
    # Ensure alcohol_kansspelen is converted to int (0 or 1) to match SQL BIT type
    flights_df['alcohol_kansspelen'] = flights_df['alcohol_kansspelen'].astype(int)

    update_query = """
    UPDATE [dbo].[tv.flights_ster]
    SET alcohol_kansspelen = :alcohol_kansspelen
    WHERE LTRIM(RTRIM(flight_id)) = :flight_id
    """

    with engine.begin() as conn:
        for index, row in flights_df.iterrows():
            # Execute the update query for each row
            conn.execute(
                text(update_query),
                {
                    "alcohol_kansspelen": row['alcohol_kansspelen'],
                    "flight_id": row['flight_id']
                }
            )
            # Optional: Confirm update
            print(f"Updated flight_id {row['flight_id']} with alcohol_kansspelen = {row['alcohol_kansspelen']}")

def process_and_update_by_year(start_year, end_year):
    query_spots = """
        SELECT *
        FROM [dbo].[tv.spots_ster]
        ORDER BY date, advertiser;
    """
    # Fetch all spots once, since they are reused across all years
    all_spots = db_utils.query_to_df(query_spots)

    for year in range(start_year, end_year + 1):
        print(f"Processing year: {year}")

        # Create the engine for each year to avoid timeout
        engine = con_utils.create_sql_alchemy_engine()

        # Query the flights for the current year
        query_flights = f"""
        SELECT *
        FROM [dbo].[tv.flights_ster]
        WHERE start_date >= '{year}-01-01' AND start_date <= '{year}-12-31'j
        """
        flights = db_utils.query_to_df(query_flights)

        if not flights.empty:
            # Update the alcohol_kansspelen column
            update_alcohol_label(flights, all_spots)

            # Update the database for this year's flights
            update_alcohol_column(flights, engine)

        print(f"Finished processing year: {year}")

        # Dispose of the engine after use to release resources
        engine.dispose()

# Loop through years 2018 to 2024 and process each year
process_and_update_by_year(2018, 2024)
